# Plant Disease Detection Using YOLO11 Classification

This notebook trains and evaluates a YOLO11 classification model on the Kaggle **New Plant Diseases Dataset**.

The project originally started as a tomato leaf disease detection idea, but this specific Kaggle dataset is organized for **image classification**, not object detection. That means the model predicts the disease class of the whole leaf image instead of drawing bounding boxes around diseased regions.

Main goals:

- download the dataset with KaggleHub
- prepare the folder structure expected by Ultralytics YOLO classification
- inspect class distribution and sample images
- fine-tune a YOLO11 classification model
- avoid overtraining with early stopping
- evaluate top-1 accuracy, top-5 accuracy, and confusion matrix
- create prediction examples for a portfolio project


## 1. Install Ultralytics

This cell installs the Ultralytics package, which provides the YOLO11 model family and the training, validation, and prediction APIs used in this notebook.


In [ ]:
# Install the Ultralytics library so we can use YOLO11 models.
# The -q flag means "quiet", which keeps the Kaggle output shorter.
!pip install -q ultralytics


## 2. Import Libraries and Check GPU

This cell imports the Python libraries used throughout the project. It also checks whether Kaggle has attached a GPU, because training deep learning models is much faster on GPU than CPU.


In [ ]:
# pathlib gives cleaner file/path handling than manually joining strings.
from pathlib import Path

# Standard utilities for randomness, file operations, and compression.
import os
import random
import shutil

# Data analysis and plotting libraries.
import pandas as pd
import matplotlib.pyplot as plt

# Pillow is used to open and display image files.
from PIL import Image

# PyTorch is the deep learning backend used by Ultralytics.
import torch

# KaggleHub downloads Kaggle datasets from code.
import kagglehub

# YOLO is the main Ultralytics class for loading, training, validating, and predicting.
from ultralytics import YOLO

# Print environment information so we know what hardware the notebook is using.
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

# If CUDA is available, print the GPU name.
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Download the Dataset with KaggleHub

This cell downloads the latest version of `vipoooool/new-plant-diseases-dataset` and stores the local dataset path in `dataset_download_path`.

KaggleHub may place files under a Kaggle input path or a local cache path, so later cells should use this returned path instead of assuming one fixed location.


In [ ]:
# Download the dataset from KaggleHub.
# This dataset is organized by class folders, so it is suitable for classification.
path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')

# Convert the returned string path into a Path object for easier path operations.
dataset_download_path = Path(path)

# Print the path so we can debug folder locations if needed.
print('Path to dataset files:', dataset_download_path)


## 4. Inspect Dataset Folders Safely

The dataset contains many thousands of image files. Printing every file with `rglob('*')` can freeze the browser, so this cell only prints top-level folders and checks the most common nested dataset paths.


In [ ]:
# Print the dataset root returned by KaggleHub.
print('Dataset download path:')
print(dataset_download_path)

# Show only top-level files/folders to avoid flooding the notebook output.
print('
Top-level files/folders:')
for p in sorted(dataset_download_path.iterdir()):
    print('-', p.name)

# This Kaggle dataset often has nested folders with very similar names.
# We test the likely dataset roots without recursively printing every image file.
print('
Common nested folders to check:')
possible_nested_roots = [
    dataset_download_path / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'new plant diseases dataset(augmented)' / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'new plant diseases dataset(augmented)',
]

# Print whether each possible root exists.
for root in possible_nested_roots:
    print(root, '->', root.exists())


## 5. Select the Train and Validation Folders

Ultralytics classification expects separate training and validation folders. This cell finds the nested dataset root that contains both `train` and `valid` folders, then stores those paths for later use.


In [ ]:
# Image extensions accepted by this notebook.
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Candidate roots based on the known folder structure of this Kaggle dataset.
dataset_root_candidates = [
    dataset_download_path / 'New Plant Diseases Dataset(Augmented)' / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'new plant diseases dataset(augmented)' / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'New Plant Diseases Dataset(Augmented)',
    dataset_download_path / 'new plant diseases dataset(augmented)',
    dataset_download_path,
]

# Find the first candidate that contains both train and valid folders.
dataset_root = None
for candidate in dataset_root_candidates:
    if (candidate / 'train').exists() and (candidate / 'valid').exists():
        dataset_root = candidate
        break

# Stop early with a clear error if the expected folders are not found.
assert dataset_root is not None, 'Could not find train/valid folders. Check the printed candidate paths above.'

# Store the real training and validation directories.
train_dir = dataset_root / 'train'
val_dir = dataset_root / 'valid'

# Class names are represented as folder names inside train/valid.
train_classes = [p for p in train_dir.iterdir() if p.is_dir()]
val_classes = [p for p in val_dir.iterdir() if p.is_dir()]

# Print a short sanity check.
print('Selected dataset_root:', dataset_root)
print('Train exists:', train_dir.exists())
print('Valid exists:', val_dir.exists())
print('Number of train classes:', len(train_classes))
print('Number of valid classes:', len(val_classes))

# Print only a few class names to keep output readable.
print('
First 10 train classes:')
for class_dir in sorted(train_classes)[:10]:
    print('-', class_dir.name)


## 6. Create a YOLO-Compatible Dataset Folder

The dataset uses `valid`, but Ultralytics classification commonly expects `val`. This cell creates a small working dataset folder with `train` and `val` entries.

It tries to use symlinks first because symlinks avoid copying thousands of images. If symlinks are not allowed, it falls back to copying the folders.


In [ ]:
# This is the clean dataset root passed into YOLO training.
WORK_DATASET = Path('/kaggle/working/plant_disease_cls_dataset')

# Remove an old working dataset if this cell is rerun.
if WORK_DATASET.exists():
    shutil.rmtree(WORK_DATASET)

# Create the working dataset root.
WORK_DATASET.mkdir(parents=True, exist_ok=True)

# Helper function: symlink when possible, copy when symlink fails.
def link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    try:
        # Symlink is fast and saves disk space.
        dst.symlink_to(src, target_is_directory=True)
    except Exception:
        # Copy is slower but works in environments where symlinks are blocked.
        shutil.copytree(src, dst)

# Create YOLO-compatible train and val folders.
link_or_copy(train_dir, WORK_DATASET / 'train')
link_or_copy(val_dir, WORK_DATASET / 'val')

# Confirm the working folder is ready.
print('YOLO classification dataset root:', WORK_DATASET)
print('Contains:', [p.name for p in WORK_DATASET.iterdir()])


## 7. Count Images per Class

This cell counts how many images exist in each class. Class balance matters because a model can become biased if some classes have far more images than others.


In [ ]:
# Count image files for every class folder in a split.
def count_images_by_class(split_dir):
    rows = []
    for class_dir in sorted(Path(split_dir).iterdir()):
        if class_dir.is_dir():
            # Count only image files, ignoring any non-image files.
            n = sum(1 for p in class_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS)
            rows.append({'class': class_dir.name, 'images': n})
    return pd.DataFrame(rows)

# Count images in train and validation splits.
train_counts = count_images_by_class(WORK_DATASET / 'train')
val_counts = count_images_by_class(WORK_DATASET / 'val')

# Display summary information.
print('Number of classes:', len(train_counts))
display(train_counts.head())
display(train_counts.describe())


## 8. Visualize Class Distribution

This chart shows the number of training images per class. It helps reveal whether the dataset is balanced or whether some classes dominate the training data.


In [ ]:
# Create a horizontal bar chart of class image counts.
plt.figure(figsize=(12, 10))
train_counts.sort_values('images').plot(
    kind='barh',
    x='class',
    y='images',
    legend=False,
    ax=plt.gca()
)

# Label the plot for readability.
plt.title('Training Images per Class')
plt.xlabel('Number of images')
plt.ylabel('Class')
plt.tight_layout()
plt.show()


## 9. Visualize Sample Training Images

Before training, it is important to look at real images from the dataset. This verifies that the folders really contain leaf images and helps you understand what visual patterns the model will learn.


In [ ]:
# Collect all class folders from the training split.
class_dirs = [p for p in (WORK_DATASET / 'train').iterdir() if p.is_dir()]

# Randomly select up to 9 classes for visualization.
sample_classes = random.sample(class_dirs, min(9, len(class_dirs)))

# Create a 3x3 image grid.
plt.figure(figsize=(12, 12))

for i, class_dir in enumerate(sample_classes):
    # Find images inside the selected class folder.
    images = [p for p in class_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS]

    # Skip empty folders safely.
    if not images:
        print('No images found in:', class_dir)
        continue

    # Pick one image at random from the class.
    img_path = random.choice(images)
    img = Image.open(img_path).convert('RGB')

    # Display the image and use the folder name as the class label.
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(class_dir.name, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()


## 10. Optional Smoke Test Training

A smoke test trains for only one epoch. It is useful for checking that the dataset path, class folders, and model setup work before spending time on a full training run.

This cell is commented out because the main training has already been run, but it is useful to keep for reproducibility.


In [ ]:
# Smoke test: one epoch only.
# Uncomment this cell if you want to quickly verify that training starts correctly.

# model = YOLO('yolo11n-cls.pt')

# smoke_results = model.train(
#     data=str(WORK_DATASET),
#     epochs=1,
#     imgsz=224,
#     batch=32,
#     project='/kaggle/working/runs',
#     name='plant_disease_yolo11_cls_smoke',
#     exist_ok=True
# )


## 11. Training with Early Stopping

This is the standard training setup. `epochs=50` is only the maximum limit. `patience=5` tells Ultralytics to stop if validation performance does not improve for 5 epochs, which helps avoid unnecessary overtraining.

Ultralytics automatically saves `best.pt`, the best model checkpoint found during training.


In [ ]:
# Full training with early stopping.
# This cell is commented out because this notebook contains your completed run.
# Uncomment it if you want to train again from scratch.

# model = YOLO('yolo11n-cls.pt')

# train_results = model.train(
#     data=str(WORK_DATASET),
#     epochs=50,        # Maximum number of epochs, not guaranteed epochs.
#     imgsz=224,        # Input image size used by the classifier.
#     batch=32,         # Number of images processed per training step.
#     patience=5,       # Stop if validation performance does not improve for 5 epochs.
#     save=True,        # Save best.pt and last.pt.
#     save_period=1,    # Save an extra checkpoint for every epoch.
#     project='/kaggle/working/runs',
#     name='plant_disease_yolo11n_cls_earlystop',
#     exist_ok=True
# )


## 12. Regularized Fine-Tuning

The validation accuracy can become very high on this dataset because the images are clean and the validation split is similar to the training split. This fine-tuning setup is more conservative: it uses a lower learning rate, dropout, weight decay, cosine learning rate scheduling, and early stopping.

The goal is not to fake a lower accuracy. The goal is to reduce overfitting risk and make the experiment more defensible.


In [ ]:
# More conservative fine-tuning experiment.
# Use this when the normal run reaches near-perfect validation accuracy very quickly.

RUN_REGULARIZED_FINETUNE = True

if RUN_REGULARIZED_FINETUNE:
    # Load pretrained YOLO11 nano classification weights.
    model = YOLO('yolo11n-cls.pt')

    # Fine-tune the model with regularization and early stopping.
    finetune_results = model.train(
        data=str(WORK_DATASET),
        epochs=50,           # Maximum epochs; early stopping may stop sooner.
        imgsz=224,           # Classification image size.
        batch=32,            # Lower this if GPU memory is limited.
        patience=5,          # Stop after 5 epochs without validation improvement.
        lr0=0.001,           # Lower starting learning rate for gentler fine-tuning.
        lrf=0.01,            # Final learning rate fraction.
        weight_decay=0.001,  # Penalizes overly large weights to reduce overfitting.
        dropout=0.2,         # Randomly drops features during training for regularization.
        cos_lr=True,         # Uses cosine learning rate schedule.
        save=True,           # Save best.pt and last.pt.
        save_period=1,       # Save every epoch checkpoint.
        project='/kaggle/working/runs',
        name='plant_disease_yolo11n_cls_regularized_finetune',
        exist_ok=True
    )
else:
    print('Regularized fine-tuning is disabled. Set RUN_REGULARIZED_FINETUNE = True if you want to run it.')


## 13. Optional: Zip the Working Folder

This optional cell can package the Kaggle working directory into one zip file. It is useful when you want to download training outputs, but it is commented out to avoid creating large files accidentally.


In [ ]:
# Optional export of the whole Kaggle working folder.
# Uncomment only if you want a full backup zip.

# from IPython.display import FileLink

# zip_path = '/kaggle/working/working_folder.zip'

# if os.path.exists(zip_path):
#     os.remove(zip_path)

# shutil.make_archive('/kaggle/working/working_folder', 'zip', '/kaggle/working')

# FileLink('/kaggle/working/working_folder.zip')


## 14. Validate the Best Model

Validation measures performance on images not used directly for training. This cell loads the selected `best.pt` checkpoint and reports top-1 and top-5 accuracy.

Top-1 accuracy means the model's first prediction is correct. Top-5 accuracy means the correct class appears somewhere in the model's five most confident predictions.


In [ ]:
# Paths to possible trained model checkpoints.
SMOKE_WEIGHTS = Path('/kaggle/working/runs/plant_disease_yolo11_cls_smoke/weights/best.pt')
FULL_WEIGHTS = Path('/kaggle/working/runs/plant_disease_yolo11n_cls_earlystop/weights/best.pt')
REGULARIZED_WEIGHTS = Path('/kaggle/working/runs/plant_disease_yolo11n_cls_regularized_finetune/weights/best.pt')

# Prefer the regularized fine-tuned model if it exists.
# Otherwise fall back to the normal early-stopping model, then smoke-test model.
if REGULARIZED_WEIGHTS.exists():
    BEST_WEIGHTS = REGULARIZED_WEIGHTS
elif FULL_WEIGHTS.exists():
    BEST_WEIGHTS = FULL_WEIGHTS
else:
    BEST_WEIGHTS = SMOKE_WEIGHTS

print('Using weights:', BEST_WEIGHTS)

# Load the trained model checkpoint.
trained_model = YOLO(str(BEST_WEIGHTS))

# Run validation on the prepared dataset.
metrics = trained_model.val(data=str(WORK_DATASET), imgsz=224)

# Print the most important classification metrics.
print('Top-1 accuracy:', metrics.top1)
print('Top-5 accuracy:', metrics.top5)


## 15. View Training Curves and Confusion Matrix

Ultralytics saves visual summaries after training. `results.png` shows training/validation curves, while the confusion matrix shows which plant disease classes are confused with each other.


In [ ]:
# The run directory is two levels above best.pt: weights/best.pt -> run folder.
run_dir = BEST_WEIGHTS.parent.parent
print('Run directory:', run_dir)

# Display common result images if they exist.
for filename in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']:
    img_path = run_dir / filename

    if img_path.exists():
        # Open the saved plot as an RGB image.
        img = Image.open(img_path).convert('RGB')

        # Show the plot inside the notebook.
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.title(filename)
        plt.axis('off')
        plt.show()
    else:
        print('Not found:', img_path)


## 16. Run Predictions on Validation Images

This cell randomly selects validation images and shows the true class beside the model's predicted class. These examples are useful for a README, report, or portfolio screenshot.


In [ ]:
# Collect validation images from all class folders.
val_images = [p for p in (WORK_DATASET / 'val').rglob('*') if p.suffix.lower() in IMAGE_EXTS]

# Select up to 9 images at random.
sample_images = random.sample(val_images, min(9, len(val_images)))

# Run model prediction. verbose=False keeps the output cleaner.
results = trained_model.predict(sample_images, imgsz=224, verbose=False)

# Create a 3x3 grid of predictions.
plt.figure(figsize=(12, 12))

for i, result in enumerate(results):
    # Open the image that was predicted.
    img = Image.open(result.path).convert('RGB')

    # Extract the top predicted class ID, name, and confidence.
    pred_id = int(result.probs.top1)
    pred_name = result.names[pred_id]
    confidence = float(result.probs.top1conf)

    # The true label is the parent folder name.
    true_name = Path(result.path).parent.name

    # Show image with true label and predicted label.
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(f'True: {true_name}
Pred: {pred_name} ({confidence:.2f})', fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()


## 17. Export Training Outputs

This cell compresses the Ultralytics `runs` folder into a zip file so the trained weights, plots, and prediction outputs can be downloaded from Kaggle.


In [ ]:
# Create a zip file containing all training runs and output artifacts.
OUTPUT_ZIP = '/kaggle/working/plant_disease_yolo11_cls_outputs'
shutil.make_archive(OUTPUT_ZIP, 'zip', '/kaggle/working/runs')

# Print the zip path so it is easy to find in Kaggle output files.
print('Created:', OUTPUT_ZIP + '.zip')
